# Granular Biomechanics Dataset Goals:

provide data for two different EMG datasets:
1) the resampled/interpolated biomech  inner joined to emg dataset where we join them and then simulate the bio data in betweeen what we have so we don't lose emg data
2) for the left joined phase data onto EMG data, so we get the phases without losing emg data (this is a just in case the interpolated data is not reliable)


In [15]:
import pandas as pd
import mysql.connector
import logging
from datetime import datetime, timedelta
import numpy as np

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_database_connection():
    """Create and return a database connection to the 'theia_pitching_db'."""
    return mysql.connector.connect(
        host="10.200.200.107",
        user="scriptuser",
        password="!!Driveline11",
        database="theia_pitching_db"
    )

def check_data_completeness(df, table_name, key_columns):
    """
    Check for missing/incomplete data in a given DataFrame.
    Logs the sum of nulls per column, unique non-null counts, and data types.
    If a 'time' column is present, calculates basic time gap statistics.
    """
    logger.info(f"\nChecking data completeness for {table_name}:")
    null_counts = df.isnull().sum()
    if null_counts.any():
        logger.warning(f"Null values found in {table_name}:")
        logger.warning(null_counts[null_counts > 0])
    else:
        logger.info("No null values found.")
    
    for col in df.columns:
        unique_count = df[col].nunique(dropna=True)
        col_type = df[col].dtype
        logger.info(f"Column '{col}': Type = {col_type}, Unique non-null values = {unique_count}")
    
    if 'time' in df.columns:
        time_stats = df.groupby('session_trial')['time'].apply(lambda x: x.diff().describe())
        logger.info(f"Time series statistics for {table_name} (averaged across sessions):\n{time_stats.mean()}")

def get_granular_time_series_data(filter_type='LAST_DAY', specific_date=None, specific_month=None):
    """
    Retrieves granular, frame-level time series data from 'theia_pitching_db'
    with flexible date filtering options and returns all key metrics for injury analysis.
    
    Updates include:
      - Enriched session filtering via an expanded CTE that joins trials, sessions, and users
        to include athlete details (name, dob, traq) as well as trial-level information.
      - Addition of a pitch_phases CTE to retrieve event markers (PKH_time, FP_v5_time, MER_time, BR_time, MAD_time).
      - Computation of pitch_phase label in the frame_level_data CTE based on event times.
      - Final output joins additional tables: energetics, force_plates, joint_forces, joint_moments, and poi
        for static pitch metrics.
    
    Parameters:
      filter_type (str): One of 'LAST_DAY', 'LAST_5_DAYS', 'LAST_MONTH', 'SPECIFIC_DATE', 'SPECIFIC_MONTH'
      specific_date (str): Date in 'YYYY-MM-DD' format (required for 'SPECIFIC_DATE')
      specific_month (str): Month in 'YYYY-MM' format (required for 'SPECIFIC_MONTH')
      
    Returns:
      A pandas DataFrame with one row per frame and all aligned metrics.
    """
    conn = get_database_connection()
    cursor = conn.cursor()
    
    # Build the dynamic filtering part of the date_filtered_sessions CTE.
    # This CTE now joins trials, sessions, and users to return rich athlete and session metadata.
    date_filter_cte = """
    WITH date_filtered_sessions AS (
        SELECT
            t.session_trial,
            t.trial,
            t.time AS trial_time,
            TIMESTAMP(s.date, t.time) AS session_datetime,  -- Combines date and time into one datetime


            t.pitch_type,
            t.handedness,
            s.date,
            s.session,
            s.level,
            s.lab,
            s.height_meters,
            s.mass_kilograms,
            u.name AS athlete_name,
            u.dob AS athlete_dob,
            u.traq AS athlete_traq
        FROM `trials` t
        JOIN `sessions` s ON t.session = s.session
        JOIN `users` u ON s.user = u.user
        WHERE 
    """
    
    # Append the appropriate date filter based on the filter_type
    if filter_type == 'LAST_DAY':
        date_filter_cte += "s.date = (SELECT MAX(date) FROM `sessions`)"
    elif filter_type == 'LAST_5_DAYS':
        date_filter_cte += "s.date >= DATE_SUB((SELECT MAX(date) FROM `sessions`), INTERVAL 4 DAY)"
    elif filter_type == 'LAST_MONTH':
        date_filter_cte += "s.date >= DATE_SUB((SELECT MAX(date) FROM `sessions`), INTERVAL 30 DAY)"
    elif filter_type == 'SPECIFIC_DATE':
        if not specific_date:
            raise ValueError("specific_date is required for SPECIFIC_DATE filter_type")
        date_filter_cte += f"s.date = '{specific_date}'"
    elif filter_type == 'SPECIFIC_MONTH':
        if not specific_month:
            raise ValueError("specific_month is required for SPECIFIC_MONTH filter_type")
        date_filter_cte += f"DATE_FORMAT(s.date, '%Y-%m') = '{specific_month}'"
    else:
        raise ValueError(f"Invalid filter_type: {filter_type}")
    
    date_filter_cte += ")"
    
    # CTE to retrieve event markers for the pitch phases
    pitch_phases_cte = """
    , pitch_phases AS (
        SELECT 
            e.session_trial,
            e.PKH_time,
            e.FP_v5_time,
            e.MER_time,
            e.BR_time,
            e.MAD_time
        FROM `events` e
        INNER JOIN date_filtered_sessions dfs 
            ON e.session_trial = dfs.session_trial
    )
    """
    
    # CTE to get frame-level joint data along with phase computation using event markers
    frame_level_data_cte = """
    , frame_level_data AS (
        SELECT 
            ja.session_trial,
            ja.time,
            -- Joint angles
            ja.shoulder_angle_x,
            ja.shoulder_angle_y,
            ja.shoulder_angle_z,
            ja.elbow_angle_x,
            ja.elbow_angle_y,
            ja.elbow_angle_z,
            ja.torso_angle_x,
            ja.torso_angle_y,
            ja.torso_angle_z,
            ja.pelvis_angle_x,
            ja.pelvis_angle_y,
            ja.pelvis_angle_z,
            
            -- Joint velocities
            jv.shoulder_velo_x,
            jv.shoulder_velo_y,
            jv.shoulder_velo_z,
            jv.elbow_velo_x,
            jv.elbow_velo_y,
            jv.elbow_velo_z,
            jv.torso_velo_x,
            jv.torso_velo_y,
            jv.torso_velo_z,
            
            -- Derived metric: trunk-pelvis dissociation
            ABS(ja.torso_angle_z - ja.pelvis_angle_z) AS trunk_pelvis_dissociation,
            
            -- Pitch phase label computed based on event times
            CASE 
                WHEN ja.time <= pp.PKH_time THEN 'Wind-Up'
                WHEN ja.time <= pp.FP_v5_time THEN 'Stride'
                WHEN ja.time <= pp.MER_time THEN 'Arm Cocking'
                WHEN ja.time <= pp.BR_time THEN 'Arm Acceleration'
                WHEN ja.time <= pp.MAD_time THEN 'Arm Deceleration'
                ELSE 'Follow Through'
            END AS pitch_phase
        FROM `joint_angles` ja
        INNER JOIN date_filtered_sessions dfs 
            ON ja.session_trial = dfs.session_trial
        INNER JOIN `joint_velos` jv 
            ON ja.session_trial = jv.session_trial 
            AND ja.time = jv.time
        LEFT JOIN pitch_phases pp 
            ON ja.session_trial = pp.session_trial
    )
    """
    
    # Build the final query that joins the frame-level data with additional metrics tables.
    query = (
        date_filter_cte +
        pitch_phases_cte +
        frame_level_data_cte +
        """
    SELECT 
        -- Athlete and Session Information
        dfs.athlete_name,
        dfs.athlete_dob,
        dfs.athlete_traq,
        dfs.height_meters,
        dfs.mass_kilograms,
        dfs.level AS athlete_level,
        dfs.date AS session_date,
        dfs.trial_time AS session_time,
        dfs.lab,
        dfs.session,
        dfs.trial,
        dfs.pitch_type,
        dfs.handedness,
        -- New ongoing timestamp: adds the frame's time offset (in seconds) to the session start datetime
        DATE_ADD(dfs.session_datetime, INTERVAL fld.time SECOND) AS ongoing_timestamp,
        

        -- Biomechanical Frame-Level Data
        fld.*,
        
        -- Energy metrics
        en.shoulder_energy_transfer,
        en.shoulder_energy_generation,
        en.elbow_energy_transfer,
        en.elbow_energy_generation,
        en.lead_knee_energy_transfer,
        en.lead_knee_energy_generation,
        
        -- Force plate data
        fp.lead_force_x,
        fp.lead_force_y,
        fp.lead_force_z,
        fp.lead_force_mag,
        fp.rear_force_x,
        fp.rear_force_y,
        fp.rear_force_z,
        fp.rear_force_mag,
        
        -- Joint forces and moments
        jf.elbow_force_x,
        jf.elbow_force_y,
        jf.elbow_force_z,
        jf.shoulder_upper_arm_force_x,
        jf.shoulder_upper_arm_force_y,
        jf.shoulder_upper_arm_force_z,
        
        jm.elbow_moment_x,
        jm.elbow_moment_y,
        jm.elbow_moment_z,
        jm.shoulder_thorax_moment_x,
        jm.shoulder_thorax_moment_y,
        jm.shoulder_thorax_moment_z,
        
        -- Static pitch metrics
        p.pitch_speed_mph,
        p.max_shoulder_internal_rotational_velo

    FROM frame_level_data fld
    LEFT JOIN `energetics` en 
        ON fld.session_trial = en.session_trial 
        AND fld.time = en.time
    LEFT JOIN `force_plates` fp 
        ON fld.session_trial = fp.session_trial 
        AND fld.time = fp.time
    LEFT JOIN `joint_forces` jf 
        ON fld.session_trial = jf.session_trial 
        AND fld.time = jf.time
    LEFT JOIN `joint_moments` jm 
        ON fld.session_trial = jm.session_trial 
        AND fld.time = jm.time
    LEFT JOIN `poi` p 
        ON fld.session_trial = p.session_trial
    -- Rejoin with date_filtered_sessions to bring in session and athlete details
    JOIN date_filtered_sessions dfs 
        ON fld.session_trial = dfs.session_trial
    ORDER BY 
        dfs.date,
        dfs.trial_time,
        fld.session_trial,
        fld.time;
    """
    )
    
    logger.info(f"Executing updated query with filter_type: {filter_type}")
    cursor.execute(query)
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    
    df = pd.DataFrame(rows, columns=columns)
    cursor.close()
    conn.close()
    
    logger.info(f"Query returned {len(df)} rows and {len(df.columns)} columns.")
    return df


if __name__ == "__main__":
    # Example usage with different filter types

    logger.info("Requesting granular time series dataset with LAST_DAY filter...")
    df_last_day = get_granular_time_series_data(filter_type='LAST_DAY')
    logger.info("LAST_DAY filter complete. Displaying head:")
    print(df_last_day.head(10))
    logger.info("Displaying pitch phase data from last day:")
    print("\nPitch phase values:")
    print(df_last_day['pitch_phase'].value_counts().sort_index())

    logger.info("Requesting granular time series dataset with LAST_5_DAYS filter...")
    df_last_5 = get_granular_time_series_data(filter_type='LAST_5_DAYS')
    logger.info("LAST_5_DAYS filter complete. Displaying head:")
    print(df_last_5.head(10))
    
    logger.info("Requesting granular time series dataset with SPECIFIC_DATE filter (2025-02-14)...")
    df_specific = get_granular_time_series_data(filter_type='SPECIFIC_DATE', specific_date='2025-02-14')
    logger.info("SPECIFIC_DATE filter complete. Displaying head:")
    print(df_specific.head(10))

    logger.info("\nChecking ongoing_timestamp column:")
    logger.info(f"Number of unique timestamps: {df_specific['ongoing_timestamp'].nunique()}")
    logger.info("\nTimestamp range:")
    logger.info(f"Earliest: {df_specific['ongoing_timestamp'].min()}")
    logger.info(f"Latest: {df_specific['ongoing_timestamp'].max()}")
    logger.info("\nNumber of null values: {df_specific['ongoing_timestamp'].isnull().sum()}")
    logger.info("\nSample of timestamps:")
    print(df_specific['ongoing_timestamp'].head())

    logger.info("\nColumn information:")
    for col in df_specific.columns:
        logger.info(f"\nColumn: {col}")
        logger.info(f"Data type: {df_specific[col].dtype}")
        logger.info(f"Number of unique values: {df_specific[col].nunique()}")
        logger.info(f"Number of null values: {df_specific[col].isnull().sum()}")
        if df_specific[col].dtype in ['object', 'category']:
            logger.info("Sample unique values:")
            print(df_specific[col].unique()[:5])
        elif df_specific[col].dtype in ['int64', 'float64']:
            logger.info("Numeric summary:")
            print(df_specific[col].describe())

    #-----------------DROPPING BECAUSE NO FORCE PLATE DATA--------------------------------
    # Drop force plate columns
    force_plate_cols = [col for col in df_specific.columns if 'force_' in col.lower()]
    logger.info(f"\nDropping {len(force_plate_cols)} force plate columns:")
    for col in force_plate_cols:
        logger.info(f"- {col}")
    df_specific = df_specific.drop(columns=force_plate_cols)
    logger.info(f"Remaining columns after drop: {len(df_specific.columns)}")


    # Uncomment below to test SPECIFIC_MONTH filter:
    # logger.info("Requesting granular time series dataset with SPECIFIC_MONTH filter (2024-02)...")
    # df_month = get_granular_time_series_data(filter_type='SPECIFIC_MONTH', specific_month='2024-02')
    # logger.info("SPECIFIC_MONTH filter complete. Displaying head:")
    # print(df_month.head(10))

    # Save specific date dataframe to parquet
    output_path = '../../data/processed/ml_datasets/granular/granular_joint_details_2025-02-14.parquet'
    logger.info(f"Saving specific date data to {output_path}")
    df_specific.to_parquet(output_path)
    logger.info("Save complete")


    


INFO:__main__:Requesting granular time series dataset with LAST_DAY filter...
INFO:__main__:Executing updated query with filter_type: LAST_DAY
INFO:__main__:Query returned 64490 rows and 67 columns.
INFO:__main__:LAST_DAY filter complete. Displaying head:
INFO:__main__:Displaying pitch phase data from last day:
INFO:__main__:Requesting granular time series dataset with LAST_5_DAYS filter...
INFO:__main__:Executing updated query with filter_type: LAST_5_DAYS


   athlete_name athlete_dob athlete_traq height_meters mass_kilograms  \
0  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
1  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
2  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
3  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
4  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
5  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
6  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
7  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
8  Matthew Baas  2009-09-22       117766        1.8800        88.4500   
9  Matthew Baas  2009-09-22       117766        1.8800        88.4500   

  athlete_level session_date    session_time           lab  session  ...  \
0   high school   2025-02-17 0 days 10:07:59  KentFacility     2781  ...   
1   high school   2025-02-17 0 days 10:07:59

INFO:__main__:Query returned 296667 rows and 67 columns.
INFO:__main__:LAST_5_DAYS filter complete. Displaying head:
INFO:__main__:Requesting granular time series dataset with SPECIFIC_DATE filter (2025-02-14)...
INFO:__main__:Executing updated query with filter_type: SPECIFIC_DATE


        athlete_name athlete_dob athlete_traq height_meters mass_kilograms  \
0  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
1  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
2  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
3  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
4  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
5  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
6  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
7  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
8  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   
9  Christian Sobotka  2004-05-29       006159        1.8000       101.1500   

  athlete_level session_date    session_time           lab  session  ...  \
0       college   2025-02-13 0 days 09:49:14  KentFacility     27

INFO:__main__:Query returned 66443 rows and 67 columns.
INFO:__main__:SPECIFIC_DATE filter complete. Displaying head:
INFO:__main__:
Checking ongoing_timestamp column:
INFO:__main__:Number of unique timestamps: 66443
INFO:__main__:
Timestamp range:
INFO:__main__:Earliest: 2025-02-14 09:50:16
INFO:__main__:Latest: 2025-02-14 17:55:57.973300
INFO:__main__:
Number of null values: {df_specific['ongoing_timestamp'].isnull().sum()}
INFO:__main__:
Sample of timestamps:
INFO:__main__:
Column information:
INFO:__main__:
Column: athlete_name
INFO:__main__:Data type: object
INFO:__main__:Number of unique values: 12
INFO:__main__:Number of null values: 0
INFO:__main__:Sample unique values:
INFO:__main__:
Column: athlete_dob
INFO:__main__:Data type: object
INFO:__main__:Number of unique values: 12
INFO:__main__:Number of null values: 0
INFO:__main__:Sample unique values:
INFO:__main__:
Column: athlete_traq
INFO:__main__:Data type: object
INFO:__main__:Number of unique values: 12
INFO:__main__:Numbe

...

INFO:__main__:Number of unique values: 56985
INFO:__main__:Number of null values: 115
INFO:__main__:Sample unique values:
INFO:__main__:
Column: elbow_moment_z
INFO:__main__:Data type: object
INFO:__main__:Number of unique values: 21660
INFO:__main__:Number of null values: 115
INFO:__main__:Sample unique values:
INFO:__main__:
Column: shoulder_thorax_moment_x
INFO:__main__:Data type: object
INFO:__main__:Number of unique values: 59801
INFO:__main__:Number of null values: 115
INFO:__main__:Sample unique values:
INFO:__main__:
Column: shoulder_thorax_moment_y
INFO:__main__:Data type: object


[None Decimal('-62.6989') Decimal('-46.5659') Decimal('-31.4717')
 Decimal('-18.4025')]
[None Decimal('3.5618') Decimal('2.6224') Decimal('1.7430')
 Decimal('0.9830')]
[None Decimal('39.0452') Decimal('29.9640') Decimal('21.6082')
 Decimal('14.5578')]


INFO:__main__:Number of unique values: 57824
INFO:__main__:Number of null values: 115
INFO:__main__:Sample unique values:
INFO:__main__:
Column: shoulder_thorax_moment_z
INFO:__main__:Data type: object
INFO:__main__:Number of unique values: 60496
INFO:__main__:Number of null values: 115
INFO:__main__:Sample unique values:
INFO:__main__:
Column: pitch_speed_mph
INFO:__main__:Data type: object
INFO:__main__:Number of unique values: 82
INFO:__main__:Number of null values: 1140
INFO:__main__:Sample unique values:
INFO:__main__:
Column: max_shoulder_internal_rotational_velo
INFO:__main__:Data type: object
INFO:__main__:Number of unique values: 108
INFO:__main__:Number of null values: 1140
INFO:__main__:Sample unique values:
INFO:__main__:
Dropping 14 force plate columns:
INFO:__main__:- lead_force_x
INFO:__main__:- lead_force_y
INFO:__main__:- lead_force_z


[None Decimal('24.8974') Decimal('17.2553') Decimal('10.2858')
 Decimal('4.5342')]
[None Decimal('-42.3231') Decimal('-31.2265') Decimal('-20.7947')
 Decimal('-11.6908')]
[Decimal('77.3000') Decimal('76.4000') Decimal('75.9000')
 Decimal('76.9000') Decimal('75.4000')]
[Decimal('3990.6834') Decimal('3880.6868') Decimal('3956.5652')
 Decimal('3950.6838') Decimal('3870.1335')]


INFO:__main__:- lead_force_mag
INFO:__main__:- rear_force_x
INFO:__main__:- rear_force_y
INFO:__main__:- rear_force_z
INFO:__main__:- rear_force_mag
INFO:__main__:- elbow_force_x
INFO:__main__:- elbow_force_y
INFO:__main__:- elbow_force_z
INFO:__main__:- shoulder_upper_arm_force_x
INFO:__main__:- shoulder_upper_arm_force_y
INFO:__main__:- shoulder_upper_arm_force_z
INFO:__main__:Remaining columns after drop: 53
INFO:__main__:Saving specific date data to ../../data/processed/ml_datasets/granular/granular_joint_details_2025-02-14.parquet
INFO:__main__:Save complete


1) merge as of example

*losing emg data trying to join to the biomech data


# Merge with EMG metrics
- Load From:
    * emg_data_parse_and_load_final.ipynb validates and parses the emg data with the timestamp added on, will add on bulk loading and kafka datastreaming so it's better for this case
Things to watch out for:
Tolerance in join difference? should it be within .0001 sec or how big would we set up tolerance to match up those different systems?
Validation to ensure we aren't losing any metrics from either


possibly this:
Alternative Approach: Using merge_asof for Nearest-Time Joins

If the datasets have timestamps that are close but not identical and you’d prefer to merge based on the nearest time (within a certain tolerance), you can use pd.merge_asof.

# Sort dataframes by datetime
df1 = df1.sort_values('datetime')
df2 = df2.sort_values('datetime')

# Perform an asof merge: This will match each row in df1 to the nearest row in df2
merged_asof = pd.merge_asof(df1, df2, on='datetime', direction='nearest', tolerance=pd.Timedelta('1s'))

Note: With merge_asof, it’s important that the data is sorted by the datetime column, and you can set a tolerance (e.g., one second) to control how far apart the matching timestamps can be.

In [7]:
import pandas as pd
import numpy as np

# -------------------------------
# Step 1: Load and Inspect DataFrames
# -------------------------------
# Load the processed biomechanical (granular joint) data.
merged_df_left = pd.read_parquet('../../data/processed/ml_datasets/granular/granular_joint_details_2025-02-14.parquet')
print("merged_df_left columns:")
print(merged_df_left.columns)
print("\nmerged_df_left sample data:")
print(merged_df_left.head())

# Load the processed EMG data.
EMG_metrics = pd.read_csv('../../data/processed/processed_pitch_data.csv')
print("\nEMG_metrics columns:")
print(EMG_metrics.columns)
print("\nEMG_metrics sample data:")
print(EMG_metrics.head())

# -------------------------------
# Step 2: Prepare Datetime Columns for Biomech Data
# -------------------------------
if 'ongoing_timestamp' in merged_df_left.columns:
    merged_df_left['ongoing_timestamp'] = pd.to_datetime(merged_df_left['ongoing_timestamp'])
    merged_df_left['datetime'] = merged_df_left['ongoing_timestamp']
else:
    raise ValueError("merged_df_left does not contain a suitable datetime column (e.g., 'ongoing_timestamp').")

print("\nBase dataset datetime sample:")
print(merged_df_left[['ongoing_timestamp', 'datetime']].head())

# -------------------------------
# Step 3: Prepare Datetime Columns in EMG_metrics
# -------------------------------
if 'Date/Time' in EMG_metrics.columns and 'Timestamp' in EMG_metrics.columns:
    EMG_metrics['Date/Time_parsed'] = pd.to_datetime(EMG_metrics['Date/Time'])
    EMG_metrics['Timestamp_parsed'] = pd.to_datetime(EMG_metrics['Timestamp'])
    # Use the parsed Timestamp as the common datetime field.
    EMG_metrics['emg_time'] = EMG_metrics['Timestamp']
    EMG_metrics['datetime'] = EMG_metrics['Timestamp_parsed']
elif 'Date/Time' in EMG_metrics.columns:
    EMG_metrics['emg_time'] = EMG_metrics['Date/Time']
    EMG_metrics['datetime'] = pd.to_datetime(EMG_metrics['Date/Time'])
elif 'Timestamp' in EMG_metrics.columns:
    EMG_metrics['emg_time'] = EMG_metrics['Timestamp']
    EMG_metrics['datetime'] = pd.to_datetime(EMG_metrics['Timestamp'])
else:
    raise ValueError("EMG_metrics does not contain a suitable datetime column (e.g., 'Date/Time' or 'Timestamp').")

print("\nmerged_df_left datetime range:", merged_df_left['datetime'].min(), "to", merged_df_left['datetime'].max())
print("EMG_metrics datetime range:", EMG_metrics['datetime'].min(), "to", EMG_metrics['datetime'].max())
print("EMG_metrics Timestamp range:", EMG_metrics['Timestamp'].min(), "to", EMG_metrics['Timestamp'].max())

# -------------------------------
# Step 4: Compute Time Steps and Sort DataFrames
# -------------------------------
merged_df_left['time_step'] = merged_df_left['datetime'].diff()
print("\nTime step differences in merged_df_left (first 10 rows):")
print(merged_df_left[['datetime', 'time_step']].head(10))

merged_df_left = merged_df_left.sort_values('datetime')
EMG_metrics = EMG_metrics.sort_values('datetime')

print("\nRow count of merged_df_left:", merged_df_left.shape[0])
print("Row count of EMG_metrics:", EMG_metrics.shape[0])

# -------------------------------
# Step 5: Inner Join of Biomech and EMG Data
# -------------------------------
# Using merge_asof to align rows within a 10ms tolerance.
# Note: merge_asof performs a left join, so we then drop rows with no EMG match.
merged_inner = pd.merge_asof(
    merged_df_left,
    EMG_metrics,
    on='datetime',
    direction='nearest',
    tolerance=pd.Timedelta('10ms')
)

# Drop rows where no matching EMG record was found (using a key column from EMG_metrics, e.g., 'Timestamp').
merged_inner = merged_inner.dropna(subset=['Timestamp'])

print("\nAfter inner join between biomech and EMG data:")
print("Row count:", merged_inner.shape[0])
print(merged_inner.head())

# -------------------------------
# Step 6: Save the Final Inner Join Dataset
# -------------------------------
output_inner_path = '../../data/processed/ml_datasets/merged_biomech_emg_inner_join_data.parquet'
merged_inner.to_parquet(output_inner_path, index=False)
print(f"\nInner join dataset saved to: {output_inner_path}")

# -------------------------------
# Step 7: Left Join of Biomech Data onto EMG_metrics (EMG Base Metrics)
# -------------------------------
# Here, we use EMG_metrics as the base and left join the biomech data from merged_df_left
# within the same 10ms tolerance.
emg_base_metrics = pd.merge_asof(
    EMG_metrics,
    merged_df_left,
    on='datetime',
    direction='nearest',
    tolerance=pd.Timedelta('10ms')
)

print("\nAfter left join of biomech data onto EMG metrics (emg_base_metrics):")
print("Row count:", emg_base_metrics.shape[0])
print(emg_base_metrics.head())
# Display column names and data types
print("\nColumns and their data types in emg_base_metrics:")
print(emg_base_metrics.dtypes)

# Basic statistics for numeric columns
print("\nBasic statistics for numeric columns:")
print(emg_base_metrics.describe())

# Check null values for each column
print("\nNull value counts per column:")
print(emg_base_metrics.isnull().sum())

# Display unique values count for categorical columns
categorical_columns = emg_base_metrics.select_dtypes(include=['object', 'category']).columns
if len(categorical_columns) > 0:
    print("\nUnique values in categorical columns:")
    for col in categorical_columns:
        print(f"\n{col}:")
        print(emg_base_metrics[col].value_counts())

# Check for infinite values
numeric_columns = emg_base_metrics.select_dtypes(include=['float64', 'int64']).columns
inf_count = emg_base_metrics[numeric_columns].isin([np.inf, -np.inf]).sum()
print("\nInfinite value counts per column:")
print(inf_count[inf_count > 0])  # Only show columns with infinite values


# -------------------------------
# Step 8: Save the EMG Base Metrics Dataset
# -------------------------------
output_emg_base_path = '../../data/processed/ml_datasets/merged_emg_base_metrics_data.parquet'
emg_base_metrics.to_parquet(output_emg_base_path, index=False)
print(f"\nEMG base metrics dataset saved to: {output_emg_base_path}")

# -------------------------------
# Step 9: Deep Analysis on the Joins
# -------------------------------
print("\n=== Deep Analysis on the Inner Join Dataset ===")
print("Inner join dataset shape:", merged_inner.shape)
print("\nNull counts in inner join dataset:")
print(merged_inner.isnull().sum())

# Check for duplicate datetime rows (which might indicate an 'explosion' in the join)
inner_duplicates = merged_inner.duplicated(subset=['datetime']).sum()
print("\nNumber of duplicate 'datetime' rows in inner join dataset:", inner_duplicates)

# Compare the inner join to the original biomech dataset
biomech_unmatched = merged_df_left.shape[0] - merged_inner.shape[0]
print("\nNumber of biomech rows with no matching EMG (dropped in inner join):", biomech_unmatched)

print("\n=== Deep Analysis on the EMG Base Metrics Dataset (Left Join) ===")
print("EMG base metrics dataset shape:", emg_base_metrics.shape)
print("\nNull counts in EMG base metrics dataset:")
print(emg_base_metrics.isnull().sum())

# For EMG base metrics, check how many rows did NOT match any biomech record.
# Assuming 'ongoing_timestamp' comes only from the biomech data.
emg_unmatched = emg_base_metrics['ongoing_timestamp'].isnull().sum()
print("\nNumber of EMG rows with no matching biomech data:", emg_unmatched)

# Check for duplicate datetime rows here as well
emg_duplicates = emg_base_metrics.duplicated(subset=['datetime']).sum()
print("\nNumber of duplicate 'datetime' rows in EMG base metrics dataset:", emg_duplicates)


merged_df_left columns:
Index(['athlete_name', 'athlete_dob', 'athlete_traq', 'height_meters',
       'mass_kilograms', 'athlete_level', 'session_date', 'session_time',
       'lab', 'session', 'trial', 'pitch_type', 'handedness',
       'ongoing_timestamp', 'session_trial', 'time', 'shoulder_angle_x',
       'shoulder_angle_y', 'shoulder_angle_z', 'elbow_angle_x',
       'elbow_angle_y', 'elbow_angle_z', 'torso_angle_x', 'torso_angle_y',
       'torso_angle_z', 'pelvis_angle_x', 'pelvis_angle_y', 'pelvis_angle_z',
       'shoulder_velo_x', 'shoulder_velo_y', 'shoulder_velo_z', 'elbow_velo_x',
       'elbow_velo_y', 'elbow_velo_z', 'torso_velo_x', 'torso_velo_y',
       'torso_velo_z', 'trunk_pelvis_dissociation', 'pitch_phase',
       'shoulder_energy_transfer', 'shoulder_energy_generation',
       'elbow_energy_transfer', 'elbow_energy_generation',
       'lead_knee_energy_transfer', 'lead_knee_energy_generation',
       'elbow_moment_x', 'elbow_moment_y', 'elbow_moment_z',
       's

2) Resample/interpolate the biomechanics dataset to join evenly onto the EMG data

In [4]:
"""
Module: emg_biomech_inner_join.py

Goal:
    To add biomech data to the EMG data after interpolating the biomech dataset 
    granular enough to provide a datapoint for each EMG metric. In addition to 
    interpolating the biomech metrics, we add an identifier column indicating 
    whether a row was interpolated (new) or originally present. We focus on the 
    inner join workflow, retaining only EMG rows that receive complete biomech data.

Usage:
    Run this module as a script to load, process, and join the datasets.
"""

import pandas as pd
import numpy as np

def load_biomech_data(biomech_path, debug=False):
    """
    Load and prepare biomechanical data from a parquet file.
    Ensures the datetime column is parsed.
    """
    df = pd.read_parquet(biomech_path)
    if 'ongoing_timestamp' not in df.columns:
        raise ValueError("Biomech data missing 'ongoing_timestamp' column.")
    df['ongoing_timestamp'] = pd.to_datetime(df['ongoing_timestamp'])
    df['datetime'] = df['ongoing_timestamp']
    if debug:
        print(f"[DEBUG] Biomech data loaded. Shape: {df.shape}")
        print(f"[DEBUG] Datetime range: {df['datetime'].min()} to {df['datetime'].max()}")
        print(f"[DEBUG] New columns: {df[['ongoing_timestamp', 'datetime']].dtypes.to_dict()}")
    else:
        print("Biomech data loaded.")
    return df

def load_emg_data(emg_path, debug=False):
    """
    Load and prepare EMG data from a CSV file.
    Parses datetime using either 'Date/Time' and/or 'Timestamp' columns.
    """
    df = pd.read_csv(emg_path)
    if 'Date/Time' in df.columns and 'Timestamp' in df.columns:
        df['Date/Time_parsed'] = pd.to_datetime(df['Date/Time'])
        df['Timestamp_parsed'] = pd.to_datetime(df['Timestamp'])
        df['emg_time'] = df['Timestamp']
        df['datetime'] = df['Timestamp_parsed']
    elif 'Date/Time' in df.columns:
        df['emg_time'] = df['Date/Time']
        df['datetime'] = pd.to_datetime(df['Date/Time'])
    elif 'Timestamp' in df.columns:
        df['emg_time'] = df['Timestamp']
        df['datetime'] = pd.to_datetime(df['Timestamp'])
    else:
        raise ValueError("EMG data missing a datetime column ('Date/Time' or 'Timestamp').")
    if debug:
        print(f"[DEBUG] EMG data loaded. Shape: {df.shape}")
        print(f"[DEBUG] Datetime range: {df['datetime'].min()} to {df['datetime'].max()}")
    else:
        print("EMG data loaded.")
    return df

def compute_time_steps(df, debug=False):
    """
    Compute time steps based on the datetime column.
    """
    df = df.sort_values('datetime').reset_index(drop=True)
    df['time_step'] = df['datetime'].diff()
    if debug:
        print(f"[DEBUG] Time steps computed. New shape: {df.shape}")
        print(f"[DEBUG] Sample time steps: {df['time_step'].head(3).tolist()}")
    return df

def sort_dataframes(biomech_df, emg_df, debug=False):
    """
    Sort both dataframes by datetime.
    """
    biomech_df = biomech_df.sort_values('datetime').reset_index(drop=True)
    emg_df = emg_df.sort_values('datetime').reset_index(drop=True)
    if debug:
        print(f"[DEBUG] Biomech data sorted. Shape: {biomech_df.shape}")
        print(f"[DEBUG] EMG data sorted. Shape: {emg_df.shape}")
    return biomech_df, emg_df

def filter_biomech_by_emg_range(biomech_df, emg_df, buffer_minutes=30, debug=False):
    """
    For each day present in the EMG dataset, filter the biomech dataset 
    so that only rows falling within the EMG min and max datetime plus a buffer 
    are kept. (We use a larger default buffer so that the biomech data covers a 
    wider range than the EMG data.)
    """
    # Add a temporary 'date' column (date-only) to both dataframes
    biomech_df = biomech_df.copy()
    emg_df = emg_df.copy()
    biomech_df['date'] = biomech_df['datetime'].dt.date
    emg_df['date'] = emg_df['datetime'].dt.date

    filtered_list = []
    for day in emg_df['date'].unique():
        emg_day = emg_df[emg_df['date'] == day]
        day_min = emg_day['datetime'].min()
        day_max = emg_day['datetime'].max()
        # Extend the time range by a buffer on both sides
        day_min_buffered = day_min - pd.Timedelta(minutes=buffer_minutes)
        day_max_buffered = day_max + pd.Timedelta(minutes=buffer_minutes)
        if debug:
            print(f"[DEBUG] Filtering biomech for day: {day}, Original EMG range: {day_min} - {day_max}, Buffered range: {day_min_buffered} - {day_max_buffered}")
        # Filter using the buffered range
        biomech_day = biomech_df[(biomech_df['datetime'] >= day_min_buffered) & 
                                 (biomech_df['datetime'] <= day_max_buffered)]
        filtered_list.append(biomech_day)
    filtered_biomech = pd.concat(filtered_list, ignore_index=True)
    # Remove the temporary date column
    filtered_biomech = filtered_biomech.drop(columns=['date'])
    if debug:
        print(f"[DEBUG] Filtered biomech data shape after applying buffered EMG date ranges: {filtered_biomech.shape}")
    return filtered_biomech

def resample_biomech_by_emg(biomech_df, emg_df, tolerance_ms=1, debug=False):
    """
    Resample and interpolate the biomech data based on EMG timestamps.
    Here, we use the entire (buffered) biomech data for each day and then upsample 
    using a high-frequency grid whose resolution is dynamically set based on the 
    tolerance_ms value. This ensures that as the tolerance is lowered, the biomech 
    data is interpolated at a finer granularity so that every EMG timestamp is exactly 
    represented.
    """
    biomech_df = biomech_df.copy()
    emg_df = emg_df.copy()
    # Group by day using the 'date' column
    biomech_df['date'] = biomech_df['datetime'].dt.date
    emg_df['date'] = emg_df['datetime'].dt.date

    # Declare categorical columns explicitly.
    categorical_cols = [
        'athlete_name', 'athlete_dob', 'athlete_traq', 'athlete_level',
        'lab', 'pitch_type', 'handedness', 'session_date'
    ]
    categorical_numeric_cols = ['session', 'trial']

    resampled_list = []
    for day in emg_df['date'].unique():
        emg_day = emg_df[emg_df['date'] == day]
        # Use all biomech data for the day
        biomech_day = biomech_df[biomech_df['date'] == day]
        if biomech_day.empty:
            if debug:
                print(f"[DEBUG] No biomech data for day {day} (even with buffering). Skipping.")
            continue

        original_index = biomech_day.index
        biomech_day = biomech_day.set_index('datetime')
        # Create a union index of the biomech timestamps and the EMG timestamps
        union_index = biomech_day.index.union(emg_day['datetime'])
        # Get the original biomech range for this day
        original_min = biomech_day.index.min()
        original_max = biomech_day.index.max()
        # Generate a high-frequency index based on the tolerance
        resolution = f"{tolerance_ms}ms"
        high_freq_index = pd.date_range(start=original_min, end=original_max, freq=resolution)
        union_index = union_index.union(high_freq_index)
        # Restrict the union index to the original biomech data's range
        union_index = union_index[(union_index >= original_min) & (union_index <= original_max)]
        if debug:
            print(f"[DEBUG] Restricted union index range for day {day}: {union_index.min()} - {union_index.max()}")

        # Reindex and interpolate
        biomech_day = biomech_day.reindex(union_index)
        is_interpolated = ~biomech_day.index.isin(original_index)
        biomech_day["is_interpolated"] = is_interpolated

        # Determine numeric columns (excluding those that should be handled categorically)
        all_numeric = set(biomech_day.select_dtypes(include=[np.number]).columns)
        numeric_cols = list(all_numeric - set(categorical_numeric_cols))
        if debug:
            print(f"[DEBUG] Numeric columns for interpolation on day {day}: {numeric_cols}")

        biomech_day[numeric_cols] = biomech_day[numeric_cols].interpolate(method='linear')

        # For categorical columns (and those numeric columns that are categorical)
        non_numeric_cols = list(
            set(biomech_day.select_dtypes(exclude=[np.number]).columns) | set(categorical_numeric_cols)
        )
        non_numeric_cols = [col for col in non_numeric_cols if col != "is_interpolated"]
        biomech_day[non_numeric_cols] = biomech_day[non_numeric_cols].ffill().bfill()

        # Extra debug: report any remaining missing values
        missing = biomech_day.isnull().sum()
        if debug and missing.sum() > 0:
            print(f"[DEBUG] Missing values after interpolation on day {day}:")
            print(missing[missing > 0])

        biomech_day = biomech_day.reset_index().rename(columns={'index': 'datetime'})
        resampled_list.append(biomech_day)
    if resampled_list:
        resampled_biomech = pd.concat(resampled_list, ignore_index=True)
    else:
        resampled_biomech = pd.DataFrame()
    if debug:
        print(f"[DEBUG] Resampled biomech data shape: {resampled_biomech.shape}")
    return resampled_biomech

def check_emg_integrity(original_emg, joined_df, join_description="join", debug=False):
    """
    Check that the joined dataframe retains all EMG rows.
    Prints a warning if the number of rows in the joined dataframe does not match
    the original EMG dataframe.
    """
    original_count = original_emg.shape[0]
    joined_count = joined_df.shape[0]
    if debug:
        print(f"[DEBUG] {join_description} - Original EMG rows: {original_count}, Joined rows: {joined_count}")
    if original_count != joined_count:
        print(f"[WARNING] EMG data integrity check failed in {join_description}: original EMG rows = {original_count}, joined rows = {joined_count}.")
    else:
        print(f"[DEBUG] EMG data integrity check passed in {join_description}: all EMG rows retained.")

def validated_inner_join(emg_df, resampled_biomech, tolerance_ms=1, debug=False):
    """
    Perform a merge_asof join (inner join) using the EMG dataset as the base,
    ensuring that each EMG row receives a valid, interpolated biomech record.
    
    Returns the validated inner join dataframe.
    """
    tolerance = pd.Timedelta(f"{tolerance_ms}ms")
    joined = pd.merge_asof(
        emg_df.sort_values('datetime'),
        resampled_biomech.sort_values('datetime'),
        on="datetime",
        direction="nearest",
        tolerance=tolerance
    )
    if debug:
        print(f"[DEBUG] Inner join (pre-dropna) shape: {joined.shape}")
    
    # Expected biomech columns: all columns from resampled biomech except 'datetime' and 'biomech_datetime'
    expected_cols = [col for col in resampled_biomech.columns if col not in ["datetime", "biomech_datetime"]]
    missing_counts = joined[expected_cols].isnull().sum()
    if debug:
        print("[DEBUG] Missing values in expected biomech columns before dropna:")
        print(missing_counts)
    
    valid_join = joined.dropna(subset=expected_cols)
    if debug:
        dropped = joined.shape[0] - valid_join.shape[0]
        print(f"[DEBUG] Inner join (post-dropna) shape: {valid_join.shape}")
        print(f"[DEBUG] Dropped {dropped} rows due to missing biomech data.")
    else:
        dropped = joined.shape[0] - valid_join.shape[0]
    
    # Compute time differences between EMG and the original biomech timestamps.
    valid_join["time_difference"] = (valid_join["datetime"] - valid_join["biomech_datetime"]).abs()
    if debug:
        print("[DEBUG] Time difference statistics (in seconds) after inner join:")
        print(valid_join["time_difference"].dt.total_seconds().describe())
    if valid_join["time_difference"].max() > tolerance:
        print(f"[WARNING] Maximum time difference ({valid_join['time_difference'].max()}) exceeds tolerance ({tolerance}).")
    else:
        print("Time differences between EMG and biomech timestamps are within tolerance.")
    
    check_emg_integrity(emg_df, valid_join, join_description="validated_inner_join", debug=debug)
    
    # Report count of EMG rows dropped by the join
    dropped_emg_count = emg_df.shape[0] - valid_join.shape[0]
    if debug:
        print(f"[DEBUG] Total EMG rows dropped after join: {dropped_emg_count}")
    
    return valid_join

def save_dataframe(df, out_path, step_name, debug=False):
    """
    Save DataFrame to a parquet file using the provided output path.
    """
    df.to_parquet(out_path, index=False)
    if debug:
        print(f"[DEBUG] {step_name} saved to: {out_path}")
    else:
        print(f"{step_name} completed and saved.")

def deep_analysis(inner_df, debug=False):
    """
    Provides a concise deep analysis on the inner join results.
    Flags any issues such as duplicate join keys or unexpected nulls.
    """
    print("\n=== Deep Analysis ===")
    print(f"Inner join shape: {inner_df.shape}")

    null_counts = inner_df.isnull().sum()
    if null_counts.sum() > 0:
        print("[WARNING] Null values found in inner join:")
        print(null_counts[null_counts > 0])
    else:
        print("No null values found in inner join.")

    dup_count = inner_df.duplicated(subset=["datetime"]).sum()
    if dup_count:
        print(f"[WARNING] Inner join has {dup_count} duplicate datetime rows.")
    else:
        print("No duplicate datetime rows in inner join.")

    print("[INFO] Summary statistics for time differences (seconds):")
    if "time_difference" in inner_df.columns:
        print(inner_df["time_difference"].dt.total_seconds().describe())

if __name__ == "__main__":
    # Specify input and output paths
    BIOMECH_PATH = "../../data/processed/ml_datasets/granular/granular_joint_details_2025-02-14.parquet"
    EMG_PATH = "../../data/processed/processed_pitch_data.csv"
    OUTPUT_DIR = "../../data/processed/ml_datasets"
    tolerance_ms = 1  # Set a low tolerance (in ms); granularity will be adjusted accordingly
    debug = True
    buffer_minutes = 30
    # ---------------- Load Data ----------------
    biomech_df = load_biomech_data(BIOMECH_PATH, debug=debug)
    emg_df = load_emg_data(EMG_PATH, debug=debug)

    # ---------------- Initial Checks ----------------
    print("\n[INFO] Performing initial biomech dataset checks...")
    if biomech_df.isnull().sum().sum() > 0:
        print("[INFO] Dropping rows with null values from biomech data...")
        biomech_df = biomech_df.dropna()
    print(f"[INFO] Biomech data now has {len(biomech_df)} rows.")

    print("\n[INFO] Performing initial EMG dataset checks...")
    if emg_df.isnull().sum().sum() > 0:
        print("[INFO] Dropping rows with null values from EMG data...")
        emg_df = emg_df.dropna()
    print(f"[INFO] EMG data now has {len(emg_df)} rows.")

    # ---------------- Compute Time Steps & Sort ----------------
    biomech_df = compute_time_steps(biomech_df, debug=debug)
    emg_df = compute_time_steps(emg_df, debug=debug)
    biomech_df, emg_df = sort_dataframes(biomech_df, emg_df, debug=debug)

    # ---------------- Filter Biomech Data by EMG Date Ranges ----------------
    filtered_biomech = filter_biomech_by_emg_range(biomech_df, emg_df, buffer_minutes=buffer_minutes, debug=debug)
    # Report how many biomech rows in the filtered data lie outside the overall EMG range.
    overall_emg_min = emg_df['datetime'].min()
    overall_emg_max = emg_df['datetime'].max()
    outside_biomech = filtered_biomech[(filtered_biomech['datetime'] < overall_emg_min) | 
                                       (filtered_biomech['datetime'] > overall_emg_max)]
    print(f"[INFO] {outside_biomech.shape[0]} biomech rows fall outside the overall EMG time range (buffered data).")

    # ---------------- Resample and Interpolate Biomech Data ----------------
    # Note: the granularity (upsampling frequency) is automatically set based on tolerance_ms.
    resampled_biomech = resample_biomech_by_emg(filtered_biomech, emg_df, tolerance_ms=tolerance_ms, debug=debug)
    # Save a copy of the original biomech datetime as 'biomech_datetime'
    resampled_biomech["biomech_datetime"] = resampled_biomech["datetime"].copy()
    # Rename all biomech columns (except key columns) with a _biomech suffix
    resampled_biomech = resampled_biomech.rename(
        columns={col: f"{col}_biomech" for col in resampled_biomech.columns 
                 if col not in ["datetime", "biomech_datetime", "is_interpolated"]}
    )
    
    # ---------------- Validate and Perform Inner Join ----------------
    validated_join = validated_inner_join(emg_df, resampled_biomech, tolerance_ms=tolerance_ms, debug=debug)
    dropped_emg = emg_df.shape[0] - validated_join.shape[0]
    print(f"[INFO] Final validated inner join has {validated_join.shape[0]} rows out of {emg_df.shape[0]} EMG rows.")
    print(f"[INFO] {dropped_emg} EMG rows were dropped during the join (likely due to lack of nearby biomech data).")
    
    final_join_path = f"{OUTPUT_DIR}/final_inner_join_emg_biomech_data.parquet"
    save_dataframe(validated_join, final_join_path, "Final validated inner join dataset", debug=debug)

    # ---------------- Deep Analysis ----------------
    deep_analysis(validated_join, debug=debug)


[DEBUG] Biomech data loaded. Shape: (66443, 54)
[DEBUG] Datetime range: 2025-02-14 09:50:16 to 2025-02-14 17:55:57.973300
[DEBUG] New columns: {'ongoing_timestamp': dtype('<M8[ns]'), 'datetime': dtype('<M8[ns]')}
[DEBUG] EMG data loaded. Shape: (134720, 41)
[DEBUG] Datetime range: 2025-02-14 11:42:22 to 2025-02-14 11:53:32.637692501

[INFO] Performing initial biomech dataset checks...
[INFO] Dropping rows with null values from biomech data...
[INFO] Biomech data now has 64867 rows.

[INFO] Performing initial EMG dataset checks...
[INFO] EMG data now has 134720 rows.
[DEBUG] Time steps computed. New shape: (64867, 55)
[DEBUG] Sample time steps: [NaT, Timedelta('0 days 00:00:00.003300'), Timedelta('0 days 00:00:00.003300')]
[DEBUG] Time steps computed. New shape: (134720, 42)
[DEBUG] Sample time steps: [NaT, Timedelta('0 days 00:00:00.004978048'), Timedelta('0 days 00:00:00.004978049')]
[DEBUG] Biomech data sorted. Shape: (64867, 55)
[DEBUG] EMG data sorted. Shape: (134720, 42)
[DEBUG] F

# 3) Left joined phases onto EMG data
* get this from just taking out EMG data plus the phases from the dataset above